In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers torch sentencepiece

In [6]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.0 MB/s eta 0:00:00


In [2]:
model_path = "/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model"

In [ ]:
import os

# Cek apakah folder tersebut benar-benar bisa diakses
path_uji = "/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model"

if os.path.exists(path_uji):
    print("✅ Folder ditemukan!")
    print("Isi folder:", os.listdir(path_uji))
else:
    print("❌ Folder TIDAK ditemukan. Cek lagi penulisan path-nya.")

✅ Folder ditemukan!
Isi folder: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Tentukan device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

# 2. Load model (Pastikan path benar dan drive sudah mount)
model_path = "/content/drive/MyDrive/ML_Project/indobert-shopee-sentiment-model"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)

    # 3. Pindahkan ke device HANYA jika model berhasil di-load
    model.to(device)
    model.eval() # Set ke mode evaluasi
    print("✅ Model berhasil dimuat ke device!")

except Exception as e:
    print(f"❌ Gagal memuat model: {e}")

Menggunakan device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model berhasil dimuat ke device!


In [7]:
from bertopic import BERTopic
import pandas as pd

path = '/content/drive/My Drive/CSV/shopee_reviews_cleaned.csv'
df = pd.read_csv(path)

# 3. Inisialisasi BERTopic (bahasa Indonesia)
topic_model = BERTopic(language="indonesian", calculate_probabilities=True, verbose=True)

# 4. Fit Model ke teks review
topics, probs = topic_model.fit_transform(df['content_cleaned'].astype(str))

# 5. Lihat Topik yang Terbentuk
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

# 6. Visualisasikan (Ini sangat keren untuk ditaruh di portofolio!)
topic_model.visualize_topics()

2026-05-11 07:10:47,566 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

2026-05-11 07:11:07,346 - BERTopic - Embedding - Completed ✓
2026-05-11 07:11:07,349 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-11 07:11:20,598 - BERTopic - Dimensionality - Completed ✓
2026-05-11 07:11:20,599 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-11 07:11:20,674 - BERTopic - Cluster - Completed ✓
2026-05-11 07:11:20,685 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-11 07:11:20,731 - BERTopic - Representation - Completed ✓


   Topic  Count                             Name  \
0     -1    113       -1_aplikasi_di_saya_shopee   
1      0    163       0_shopee_belanja_dan_mudah   
2      1    159       1_di_pengiriman_tidak_saya   
3      2     72        2_iklan_aplikasi_di_video   
4      3     54               3_ga_lama_sampe_di   
5      4     51  4_aplikasi_bagus_belanja_sangat   
6      5     38     5_mantap_keluar_bgusssss_lee   
7      6     37   6_bagus_sangat_bermanfaat_baik   
8      7     33                7_bgus_nan_mb_bgt   
9      8     28               8_ok_oke_okey_wort   

                                      Representation  \
0  [aplikasi, di, saya, shopee, ini, indonesia, t...   
1  [shopee, belanja, dan, mudah, harga, banyak, s...   
2  [di, pengiriman, tidak, saya, kurir, paket, ud...   
3  [iklan, aplikasi, di, video, ganggu, saya, lai...   
4  [ga, lama, sampe, di, krn, nya, bisa, udh, jel...   
5  [aplikasi, bagus, belanja, sangat, membantu, y...   
6  [mantap, keluar, bgusssss, lee, 

In [8]:
# Mendapatkan info topik (Topic, Count, Name, Representation)
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,113,-1_aplikasi_di_saya_shopee,"[aplikasi, di, saya, shopee, ini, indonesia, t...",[online mall terbaik aqohhh sementara ini buat...
1,0,163,0_shopee_belanja_dan_mudah,"[shopee, belanja, dan, mudah, harga, banyak, s...","[puas belanja di shopee, shopee belanja lebih ..."
2,1,159,1_di_pengiriman_tidak_saya,"[di, pengiriman, tidak, saya, kurir, paket, ud...",[untuk shopeenya udah bagus menurutku tapi unt...
3,2,72,2_iklan_aplikasi_di_video,"[iklan, aplikasi, di, video, ganggu, saya, lai...",[iklan lu tuh ganggu anjing udah syukur gw dow...
4,3,54,3_ga_lama_sampe_di,"[ga, lama, sampe, di, krn, nya, bisa, udh, jel...",[bagus si sbnrnya cmn skrg suka kesel grgr sho...
5,4,51,4_aplikasi_bagus_belanja_sangat,"[aplikasi, bagus, belanja, sangat, membantu, y...","[aplikasi yang sangat membantu, aplikasi bagus..."
6,5,38,5_mantap_keluar_bgusssss_lee,"[mantap, keluar, bgusssss, lee, iku, lanjutkan...","[mantap, mantap, mantap]"
7,6,37,6_bagus_sangat_bermanfaat_baik,"[bagus, sangat, bermanfaat, baik, biasa, bergu...","[sangat bagus, sangat bagus, sangat bagus]"
8,7,33,7_bgus_nan_mb_bgt,"[bgus, nan, mb, bgt, lemot, baguus, hapesaya, ...","[bgus, bgus, skrg mb ny nambah besar yadi hape..."
9,8,28,8_ok_oke_okey_wort,"[ok, oke, okey, wort, okeh, lanjutt, bgt, mant...","[ok, ok, ok]"


In [9]:
# Simpan ke folder data proyekmu
topic_info.to_csv('/content/drive/My Drive/CSV/bertopic_results.csv', index=False)

print("✅ Informasi topik telah disimpan!")

✅ Informasi topik telah disimpan!


In [10]:
df['Topic'] = topics
df['Topic']

,Topic
0,2
1,8
2,7
3,12
4,0
...,...
955,13
956,2
957,1
958,3


In [14]:
# Jika label 1 = Positif, 0 = Negatif
df['Sentiment_Label'] = df['label'].map({1: 'Positif 😊', 0: 'Negatif 😡'})
df['Sentiment_Label']

,Sentiment_Label
0,Negatif 😡
1,Positif 😊
2,Negatif 😡
3,Negatif 😡
4,Positif 😊
...,...
955,Positif 😊
956,Negatif 😡
957,Negatif 😡
958,Positif 😊


In [15]:
#filter topik -1 (outlier) agar fokus
df_filtered = df[df['Topic'] != -1]

In [16]:
# 3. Buat pivot table: Baris = Topik, Kolom = Sentimen
sentiment_per_topic = df_filtered.groupby(['Topic', 'Sentiment_Label']).size().unstack(fill_value=0)

# 4. Ambil nama asli topiknya (bukan cuma angka)
topic_names = topic_model.get_topic_info()[['Topic', 'Name']]
sentiment_per_topic = sentiment_per_topic.merge(topic_names, on='Topic')

In [18]:
print(sentiment_per_topic.columns)

Index(['Topic', 'Negatif 😡', 'Positif 😊', 'Name'], dtype='object')


In [20]:
# Simpan ke folder data proyekmu
sentiment_per_topic.to_csv('/content/drive/My Drive/CSV/shopee_sentiment_per_topic.csv', index=False)

print("✅ Informasi topik telah disimpan!")

✅ Informasi topik telah disimpan!
